In [2]:
urls = [
    "https://vk.ru/andrew.kirillov",
"https://m.vk.ru/bignigga",
"https://vk.ru/y.doyneko",
"https://vk.ru/saulinivan",
"https://vk.ru/dimitr101",
"https://vk.ru/ku_ku_riz",
"https://vk.ru/medvediano",
"https://vk.ru/flashik12",
"https://vk.ru/0pupok00",
"https://vk.ru/ryosuk3",
"https://vk.ru/id139159408",
"https://vk.ru/dedvnutree",

]


def extract_user_id(url):
 part = url.rstrip("/").split("/")[-1]

 if part.startswith("id") and part[2:].isdigit():
  return(part[2:])

 return(part)

identifiers = []

for url in urls:
  identifiers.append(extract_user_id(url))

print(identifiers)

TOKEN = "29d37f6a29d37f6a29d37f6afd2a973b52229d329d37f6a4310e4115d5c77ed783eb731"
VERSION = "5.199"

import requests

response = requests.get(
    "https://api.vk.com/method/users.get",
    params={
        "users_id": ",".join(identifiers),

    }
)

response = requests.get(
    "https://api.vk.com/method/users.get",
    params={
        "user_ids": ",".join(identifiers),
        "access_token": TOKEN,
        "v": VERSION,
    }
)

data = response.json()

users = {}

for user in data['response']:
  users[user["id"]] = {
      "first_name": user["first_name"],
      "last_name": user["last_name"]
  }

group_ids = set(users.keys())

friends_by_user = {}

for user_id in group_ids:

  response = requests.get(
      "https://api.vk.com/method/friends.get",
      params={
          "user_id": user_id,
          "access_token": TOKEN,
          "v": VERSION,
      }
  )

  friends_data = response.json()

  if "error" in friends_data:
    error = friends_data["error"]
    print(
        f"Не удалось получить друзей пользователя {user_id}: "
        f"{error['error_code']} — {error['error_msg']}"
    )

  else:
    friends = friends_data['response']['items']
    friends_by_user[user_id] = friends
    print(f"Друзей у {user_id}:", len(friends))


# for user_id, friends in friends_by_user.items():
#   print(user_id, "=>", len(friends), "друзей")






level_1 = set()

for friends in friends_by_user.values():
  level_1.update(friends)

print("Друзей 1-го уровня:", len(level_1))



friends_of_friends = {}

for friend_id in level_1:

  response = requests.get(
        "https://api.vk.com/method/friends.get",
        params={
            "user_id": friend_id,
            "access_token": TOKEN,
            "v": VERSION
        }
  )

  friends_data = response.json()

  if "error" in friends_data:
    error = friends_data["error"]
    # print(
    #     f"Не удалось получить друзей пользователя {friend_id}: "
    #     f"{error['error_code']} — {error['error_msg']}"
    # )

  else:
    friends = friends_data['response']['items']
    friends_of_friends[friend_id] = friends


level_2 = set()

for friends in friends_of_friends.values():
  level_2.update(friends)

print("Людей второго уровня:", len(level_2))

level_2_new = level_2 - level_1 - group_ids

print("Новых людей:", len(level_2_new))

['andrew.kirillov', 'bignigga', 'y.doyneko', 'saulinivan', 'dimitr101', 'ku_ku_riz', 'medvediano', 'flashik12', '0pupok00', 'ryosuk3', '139159408', 'dedvnutree']


<class 'requests.exceptions.ConnectionError'>: ('Connection aborted.', HTTPException('Failed to fetch'))